## NEW MODEL

In [2]:
# working code
import os
import json
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import datetime
import requests
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from core.risk_service import RiskService
import yfinance as yf
import re


# ===============================
# 🧱 1. Knowledge Base (Memory)
# ===============================
class KnowledgeBase:
    def __init__(self, path="financial_kb.json"):
        self.path = path
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.embeddings = np.empty((0, 384), dtype="float32")
        self.texts = []
        self.index = faiss.IndexFlatL2(384)

        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            self.texts = data.get("texts", [])
            if len(self.texts) > 0:
                self.embeddings = np.array(data["embeddings"], dtype="float32")
                self.index.add(self.embeddings)

    def add_entry(self, text):
        emb = self.encoder.encode([text])
        self.embeddings = np.vstack([self.embeddings, emb])
        self.index.add(emb)
        self.texts.append(text)
        self.save()

    def retrieve(self, query, k=3):
        if len(self.texts) == 0:
            return []
        q_emb = self.encoder.encode([query])
        D, I = self.index.search(q_emb, k)
        return [self.texts[i] for i in I[0]]

    def save(self):
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump({"texts": self.texts, "embeddings": self.embeddings.tolist()}, f, indent=4)
        print(" Knowledge base updated.")


# =================================
# 🌍 2. Financial Data Fetcher (Finnhub)
# =================================
class FinancialDataFetcher:
    def __init__(self, finhub_api_key=None):
        self.finhub_key = finhub_api_key
    def get_historical_prices(self, ticker, days=180):

        try:
            data = yf.download(
                ticker,
                period=f"{days}d",
                interval="1d",
                progress=False
            )

            if data.empty:
                return None

            return data["Close"]

        except Exception:
            return None

    def get_stock_price(self, ticker):
        """Fetch current live stock price and daily change."""
        if not self.finhub_key:
            return "No Finnhub API key provided."

        try:
            url = f"https://finnhub.io/api/v1/quote?symbol={ticker.upper()}&token={self.finhub_key}"
            r = requests.get(url)
            data = r.json()

            current = data.get("c")
            prev_close = data.get("pc")
            if not current or not prev_close:
                return f"Sorry, I couldn’t fetch live data for {ticker.upper()}."

            change = current - prev_close
            pct = (change / prev_close) * 100
            direction = "up" if change > 0 else "down" if change < 0 else "unchanged"
            return f"The current price of {ticker.upper()} is ${current:.2f} ({pct:+.2f}% {direction} from yesterday)."
        except Exception as e:
            return f"Error fetching data for {ticker.upper()}: {e}"

    def get_stock_news(self, ticker, limit=5):
        """Fetch recent company news with URL and date."""
        if not self.finhub_key:
            return "No Finnhub API key provided."
        try:
            today = datetime.date.today()
            past = today - datetime.timedelta(days=5)
            url = (
                f"https://finnhub.io/api/v1/company-news?symbol={ticker.upper()}&from={past}&to={today}&token={self.finhub_key}"
            )
            r = requests.get(url)
            data = r.json()

            if isinstance(data, list) and len(data) > 0:
                headlines = []
                for i, item in enumerate(data[:limit], start=1):
                    headline = item.get("headline", "No title")
                    source = item.get("source", "Unknown")
                    datetime_ts = item.get("datetime", None)
                    date = datetime.datetime.fromtimestamp(datetime_ts).strftime("%Y-%m-%d") if datetime_ts else "N/A"
                    url_link = item.get("url", "#")
                    headlines.append(
                        {
                            "title": headline,
                            "url": url_link,
                            "source": source,
                            "date": date
                        }
                    )

                # Return structured list to be formatted by caller
                return headlines
            else:
                return []
        except Exception as e:
            print(f"Error fetching news for {ticker.upper()}: {e}")
            return []

    def get_market_impact_news(
        self,
        limit_per_category=3,
        summarize=False,
        model=None,
        tokenizer=None,
        category_filter=None
    ):
        """
        Fetch and categorize global news that can affect financial markets.
        If category_filter is provided (e.g., "energy"), only that category is shown.
        Returns formatted string.
        """
        if not self.finhub_key:
            return "No Finnhub API key provided. Can't fetch live market-impacting news."

        try:
            url = f"https://finnhub.io/api/v1/news?category=general&token={self.finhub_key}"
            r = requests.get(url)
            data = r.json()

            if not isinstance(data, list) or len(data) == 0:
                return "No recent global news available."

            # Category filters
            filters = {
                "business": ["inflation", "market", "stocks", "trade", "interest rate", "federal", "growth", "gdp", "imf"],
                "politics": ["election", "president", "policy", "congress", "geopolitical", "sanction", "minister", "parliament"],
                "energy": ["oil", "opec", "gas", "energy", "renewable", "power", "crude"],
                "technology": ["ai", "technology", "chip", "semiconductor", "software", "nvidia", "google", "apple"],
                "environment": ["climate", "carbon", "esg", "green", "sustainability", "environment", "weather"],
                "geopolitics": ["china", "india", "us", "war", "ukraine", "conflict", "middle east", "russia"]
            }

            # Normalize category filter
            if category_filter and category_filter.lower() in filters:
                category_filter = category_filter.lower()
            else:
                category_filter = None

            # Classify each article
            categorized = {key: [] for key in filters.keys()}
            for item in data:
                headline = item.get("headline", "")
                lower = headline.lower()
                for cat, words in filters.items():
                    if any(word.lower() in lower for word in words):
                        categorized[cat].append(item)
                        break

            # If specific category requested
            if category_filter:
                categories_to_display = [category_filter]
            else:
                # Show all categories that have items
                categories_to_display = [key for key, val in categorized.items() if val]

            # Format output
            final_sections = []
            for cat in categories_to_display:
                items = categorized.get(cat, [])
                if not items:
                    continue
                section_lines = [f" **{cat.title()} News:**"]
                for i, item in enumerate(items[:limit_per_category], start=1):
                    title = item.get("headline", "No title")
                    url_link = item.get("url", "#")
                    source = item.get("source", "Unknown")
                    dt = item.get("datetime", None)
                    date = datetime.datetime.fromtimestamp(dt).strftime("%Y-%m-%d") if dt else "N/A"

                    summary = ""
                    if summarize and model and tokenizer:
                        text = f"Summarize the financial impact of this news headline:\n{title}"
                        inputs = tokenizer(text, return_tensors="pt", truncation=True)
                        outputs = model.generate(**inputs, max_length=60)
                        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

                    entry = f"{i}️⃣ [{title}]({url_link})\n🗞 {source} | 📅 {date}"
                    if summary:
                        entry += f"\n Impact: {summary}"
                    section_lines.append(entry)
                final_sections.append("\n".join(section_lines))

            if not final_sections:
                return f"No relevant {category_filter or 'market'} news found."

            return "\n\n".join(final_sections)

        except Exception as e:
            return f"Error fetching market-impacting news: {e}"
    def get_ohlc_data(self, ticker, period="6mo"):
        try:
            data = yf.download(ticker, period=period, interval="1d", progress=False)
            if data.empty:
                return None
            return data
        except Exception:
            return None
    def calculate_atr(self, ticker, window=14):
        data = self.get_ohlc_data(ticker)

        if data is None or len(data) < window:
            return None

        high_low = data['High'] - data['Low']
        high_close = abs(data['High'] - data['Close'].shift())
        low_close = abs(data['Low'] - data['Close'].shift())

        true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)

        atr = true_range.rolling(window=window).mean().iloc[-1]

        return float(atr)


    def get_company_and_related_news(self, ticker, company_names=None, limit=5, summarize=False, model=None, tokenizer=None):
        """
        Returns formatted string combining:
         - Finnhub company-news (company endpoint)
         - General news filtered by presence of company name or keywords related to the company
        company_names can be a list of strings to search for in headlines (e.g. ['Apple','AAPL'])
        """
        if not self.finhub_key:
            return "No Finnhub API key provided."

        # --- Step 1: company-specific news (structured) ---
        company_news_items = self.get_stock_news(ticker, limit=limit)  # returns list of dicts (or empty)

        # --- Step 2: pull general headlines and filter by company names / keywords ---
        try:
            url = f"https://finnhub.io/api/v1/news?category=general&token={self.finhub_key}"
            r = requests.get(url)
            general = r.json()
            related = []
            if isinstance(general, list):
                # build search tokens
                tokens = []
                if company_names:
                    tokens += [t.lower() for t in company_names if t]
                # also add ticker symbol
                tokens.append(ticker.lower())

                for item in general:
                    headline = item.get("headline", "")
                    lower = headline.lower()
                    if any(tok in lower for tok in tokens):
                        related.append({
                            "title": headline,
                            "url": item.get("url", "#"),
                            "source": item.get("source", "Unknown"),
                            "date": datetime.datetime.fromtimestamp(item.get("datetime")).strftime("%Y-%m-%d") if item.get("datetime") else "N/A"
                        })

            # --- Step 3: merge & deduplicate (title-based) ---
            merged = []
            seen_titles = set()

            # helper to add structured items
            def add_item(it):
                title = it.get("title")
                if title and title not in seen_titles:
                    seen_titles.add(title)
                    merged.append(it)

            # add company-specific first
            if isinstance(company_news_items, list):
                for it in company_news_items:
                    add_item(it)

            # add related general news
            for it in related:
                add_item(it)

            # If no results at all, return helpful message
            if len(merged) == 0:
                return f"No recent news found for {ticker.upper()}."

            # --- Step 4: Format & optionally summarize ---
            output_lines = [f" Combined news for {ticker.upper()}:"]
            for i, item in enumerate(merged[:limit], start=1):
                title = item.get("title") or item.get("headline") or "No title"
                url_link = item.get("url", "#")
                source = item.get("source", "Unknown")
                date = item.get("date", "N/A")
                entry = f"{i}️⃣ [{title}]({url_link})\n    🗞 Source: {source} |  {date}"

                # optional LLM impact summary when requested
                if summarize and model and tokenizer:
                    text = f"Summarize the financial impact (1 line) of this headline:\n{title}"
                    inputs = tokenizer(text, return_tensors="pt", truncation=True)
                    outputs = model.generate(**inputs, max_length=60)
                    summ = tokenizer.decode(outputs[0], skip_special_tokens=True)
                    entry += f"\n     Impact: {summ}"

                output_lines.append(entry)

            return "\n\n".join(output_lines)

        except Exception as e:
            return f"Error fetching related company news for {ticker.upper()}: {e}"


# ===========================
# 🧩 3. Intent Classifier
# ===========================
class IntentClassifier:
    def __init__(self):
        print("Loading Intent Classifier...")
        self.classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
        self.labels = [
            "get stock price",
            "get company news",
            "get global financial news",
            "get investment advice",
            "general financial question"
        ]

    def classify(self, text):
        result = self.classifier(text, self.labels)
        intent = result["labels"][0]
        score = result["scores"][0]
        return intent, score


# ==============================
# 🧠 4. Financial Advisor (LLM)
# ==============================
class FinancialAdvisor:
    def __init__(self, knowledge_base, model_name="google/flan-t5-large", max_history=3):
        print("Loading Financial Advisor Model...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.kb = knowledge_base
        self.history = []
        self.max_history = max_history

    def generate_advice(self, question, beginner_mode=False):
        retrieved = self.kb.retrieve(question)
        context = "\n".join(retrieved)

        prompt = f"""
    You are a professional financial advisor.

    Rules:
    - Give factual, concise answers.
    - Do NOT hallucinate numbers.
    - If unsure, say "I don't have enough data".
    - Prefer explaining concepts clearly.

    Context:
    {context}

    User Question:
    {question}

    Answer:
    """

        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True)
        outputs = self.model.generate(**inputs, max_length=200)

        reply = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        return reply.strip()


# ==========================
# 🤖 5. FinanceBot (Main)
# ==========================
class FinanceBot(FinancialAdvisor):
    def __init__(self, knowledge_base, data_fetcher, model_name="google/flan-t5-large"):
        super().__init__(knowledge_base, model_name)
        self.fetcher = data_fetcher
        

        # 🆕 Risk-aware modules
        self.risk_service = RiskService()

        self.last_intent = None
        self.last_target = None
        self.intent_clf = IntentClassifier()
        self.last_intent = None
        self.last_target = None

        # Map common names for company -> (ticker, company display names / keywords)
        self.company_map = {
            "tesla": ("TSLA", ["Tesla", "TSLA"]),
            "apple": ("AAPL", ["Apple", "AAPL", "iPhone", "Cupertino"]),
            "microsoft": ("MSFT", ["Microsoft", "MSFT", "Azure"]),
            "google": ("GOOGL", ["Google", "Alphabet", "GOOGL"]),
            "amazon": ("AMZN", ["Amazon", "AMZN", "Prime"]),
            "meta": ("META", ["Meta", "META", "Facebook"]),
            "nvidia": ("NVDA", ["Nvidia", "NVDA", "GPU"]),
            "reliance": ("RELIANCE.NS", ["Reliance", "RIL"]),
            "tata": ("TATAMOTORS.NS", ["Tata", "Tata Motors", "TATAMOTORS"])
        }

    def understand_query(self, question):
        q = question.lower()
        intent, score = self.intent_clf.classify(question)

        if any(k in q for k in ["buy", "long"]):
            return "trade"
        if any(k in q for k in ["change", "difference", "up", "down", "increase", "decrease"]):
            return "get stock price"
        elif "news" in q or "headlines" in q or "global" in q:
            # keep company news separate when company name present
            return "get company news" if any(name in q for name in self.company_map.keys()) else "get global financial news"
        elif "market" in q or "trend" in q:
            return "get global financial news"
        
        if "portfolio" in q:
            return "portfolio"

        if "regime" in q:
            return "regime"

        if "forecast" in q:
            return "forecast"

        if "survival" in q or "ruin" in q:
            return "survival"

        if "risk" in q:
            return "risk"

        return intent
    

    

    def get_sentiment_for(self, ticker):
        news_items = self.fetcher.get_stock_news(ticker, limit=5)

        if not news_items:
            return 0.0

        # simple proxy: positive if many headlines, negative if few
        return np.clip(len(news_items) / 5 - 0.5, -1, 1)
    def extract_trade_params(self, text):
        

        text = text.lower().replace(",", " ")

        def parse_amount(val):
            val = val.strip()

            if "lakh" in val:
                return float(val.replace("lakh", "").strip()) * 100000
            if "k" in val:
                return float(val.replace("k", "").strip()) * 1000

            return float(val)

        # 🔥 Flexible parsing (handles real user language)
        entry_match = re.search(r"(buy|entry|at)\s*(\d+\.?\d*)", text)
        stop_match = re.search(r"(stop loss|sl|stop)\s*(\d+\.?\d*)", text)
        target_match = re.search(r"(target|tp)\s*(\d+\.?\d*)", text)

        # 🔥 Capital parsing (supports lakh / k)
        account_match = re.search(r"(capital|account)[^\d]*(\d+\.?\d*\s*(lakh|k)?)", text)

        entry = float(entry_match.group(2)) if entry_match else None
        stop = float(stop_match.group(2)) if stop_match else None
        target = float(target_match.group(2)) if target_match else None

        account = None
        if account_match:
            account = parse_amount(account_match.group(2))

        return entry, stop, target, account

            
    def answer(self, question):
        intent = self.understand_query(question)
        q_lower = question.lower()

        # find company if mentioned
        target = None
        target_company_keywords = None
        for name, (ticker, keywords) in self.company_map.items():
            if name in q_lower:
                target = ticker
                target_company_keywords = keywords
                break
        if not target and self.last_target:
            target = self.last_target
 
         # ---------- TRADE RISK ----------
        if any(word in q_lower for word in ["buy", "long"]):

            # 🔥 Ensure ticker exists
            if not target:
                return "Please specify which stock (e.g., Tesla, Apple)."

            entry, stop, target_price, account = self.extract_trade_params(question)

            # 🔥 DEBUG (VERY IMPORTANT)
            print("\n===== DEBUG TRADE INPUT =====")
            print("Ticker:", target)
            print("Entry:", entry)
            print("Stop:", stop)
            print("Target:", target_price)
            print("Capital:", account)
            print("=============================\n")

            # ---- validation ----
            missing = []
            if entry is None:
                missing.append("entry price")
            if stop is None:
                missing.append("stop loss")
            if target_price is None:
                missing.append("target")
            if account is None:
                missing.append("account size")

            if missing:
                return f"""
        Missing inputs: {', '.join(missing)}

        Example:
        Buy Tesla at 250 stop loss 230 target 270 capital 1 lakh
        """

            # ---- fetch data ----
            price_series = self.fetcher.get_historical_prices(target)

            if price_series is None:
                return f"Not enough data for {target}"

            # ---- FULL RISK PIPELINE ----
            ctx = self.risk_service.full_analysis(
                ticker=target,
                price_series=price_series,
                trade_input={
                    "entry_price": entry,
                    "stop_loss": stop,
                    "take_profit": target_price,
                    "account_size": account
                }
            )

            return ctx.explanation
        # ---------- VOLATILITY REGIME ----------
        # ---------- VOLATILITY REGIME ----------
        if "regime" in q_lower and target:

            price_series = self.fetcher.get_historical_prices(target)

            if price_series is None:
                return f"Not enough data to detect regime for {target}."

            result = self.risk_service.evaluate_regime(price_series)

            regime = result['regime']
            current_vol = result['current_vol']
            low_th = result['low_threshold']
            high_th = result['high_threshold']

            if regime == "low_vol":
                meaning = "lower-than-usual market volatility"
                implication = "larger position sizes may be safer, tighter stop-loss possible"
            elif regime == "high_vol":
                meaning = "higher-than-usual market instability"
                implication = "reduce position size, use wider stop-loss, expect sharp moves"
            else:
                meaning = "typical volatility conditions"
                implication = "standard risk management applies"

            return f"""
        Volatility Regime for {target}:

        - Current Volatility: {current_vol*100:.2f}% daily
        - Historical Thresholds:
        • Low: {low_th*100:.2f}%
        • High: {high_th*100:.2f}%
        - Regime: {regime.replace('_', ' ').title()}

        Interpretation:
        - Current volatility is {'below' if current_vol < low_th else 'above' if current_vol > high_th else 'within'} historical range.
        - This indicates {meaning}.

        Implication:
        - {implication}
        """
        # ---------- FORECAST RISK ----------
        if "forecast risk" in q_lower and target:

            price_series = self.fetcher.get_historical_prices(target)

            if price_series is None:
                return f"Not enough historical data for {target}."

            # Placeholder — replace with your TFT output
            predicted_prices = price_series.tail(30).values  # temporary fallback

            result = self.risk_service.evaluate_forecast_asset(
                target,
                price_series,
                predicted_prices
            )

            fm = result["forecast_metrics"]

            return f"""
        Forecast Integrated Risk for {target}:
        - Historical Volatility: {fm['historical_vol']:.6f}
        - Forecast Volatility: {fm['forecast_vol']:.6f}
        - Integrated Risk Score: {fm['forecast_integrated_risk']:.6f}
        """

        # ---------- RISK CONTRIBUTION ----------
        if "risk contribution" in q_lower:

            # You must define portfolio somewhere properly later
            # For now demo with single target
            if not target:
                return "Please specify assets for portfolio analysis."

            price_series = self.fetcher.get_historical_prices(target)
            if price_series is None:
                return "Not enough data."

            price_dict = {target: price_series}
            weights = {target: 1.0}

            result = self.risk_service.evaluate_risk_contribution(
                price_dict,
                weights
            )

            reply = "Portfolio Risk Contribution:\n"
            for asset, values in result.items():
                reply += f"{asset}: {values['risk_percentage']*100:.2f}% of total risk\n"

            return reply

        
        # ---------- CAPITAL SURVIVAL ----------
        # ---------- CAPITAL SIMULATION ----------
        if "capital" in q_lower or "ruin" in q_lower or "survival" in q_lower:

            if not target:
                return "Please specify a stock (e.g., Tesla, Apple) for capital simulation."

            # 🔥 Extract capital + trades from text
            capital_match = re.search(r"(capital|account)[^\d]*(\d+\.?\d*\s*(lakh|k)?)", q_lower)
            trades_match = re.search(r"(trades|steps)\s*(\d+)", q_lower)

            def parse_amount(val):
                val = val.strip()
                if "lakh" in val:
                    return float(val.replace("lakh", "").strip()) * 100000
                if "k" in val:
                    return float(val.replace("k", "").strip()) * 1000
                return float(val)

            initial_capital = 100000
            n_trades = 50

            if capital_match:
                initial_capital = parse_amount(capital_match.group(2))

            if trades_match:
                n_trades = int(trades_match.group(2))

            # 🔥 Fetch data
            price_series = self.fetcher.get_historical_prices(target)

            if price_series is None or len(price_series) < 30:
                return f"Not enough data for {target}"

            # 🔥 RUN FULL PIPELINE (THIS IS KEY)
            ctx = self.risk_service.full_analysis(
                ticker=target,
                price_series=price_series,
                capital_input={
                    "initial_capital": initial_capital,
                    "n_trades": n_trades,
                    "simulations": 500
                }
            )

            cap = ctx.capital

            capital_change = cap['median_final_capital'] - initial_capital
            return_pct = (cap['median_final_capital'] / initial_capital - 1) * 100
            dd = cap.get('avg_drawdown', 0)

            if dd > 0.25:
                risk_label = "High Capital Risk"
            elif dd > 0.15:
                risk_label = "Moderate Risk"
            else:
                risk_label = "Low Risk"

            return f"""
            Capital Risk Outlook for {target}:

            - Initial Capital: ${initial_capital:,.0f}
            - Number of Trades: {n_trades}

            Performance:
            - Median Final Capital: ${cap['median_final_capital']:,.2f}
            - Expected Capital Change: ${capital_change:,.2f}
            - Expected Return: {return_pct:.2f}%

            Risk Metrics:
            - Average Drawdown: {dd*100:.2f}%
            - Risk per Trade: {cap['risk_per_trade']*100:.2f}%

            Risk Assessment:
            - {risk_label}

            Interpretation:
            - Strategy shows positive expected return but with significant drawdown risk
            - Higher drawdown implies greater downside exposure
            - Evaluate whether return justifies the level of risk taken
            """
        
        
        # ---------- PORTFOLIO ANALYSIS ----------
        # ---------- PORTFOLIO ANALYSIS ----------
        if "portfolio" in q_lower:

            try:
                clean_q = q_lower.replace(",", " ")
                tokens = clean_q.split()

                # 🔥 Company name → ticker mapping
                name_to_ticker = {
                    "nvidia": "NVDA",
                    "apple": "AAPL",
                    "tesla": "TSLA",
                    "microsoft": "MSFT"
                }

                # -------- STEP 1: Parse holdings --------
                holdings = {}

                for i in range(len(tokens) - 1):
                    if tokens[i].isalpha() and re.match(r"\d+\.?\d*", tokens[i+1]):
                        raw = tokens[i]
                        ticker = name_to_ticker.get(raw, raw).upper()
                        amount = float(tokens[i+1])
                        holdings[ticker] = amount

                if len(holdings) < 2:
                    return "Please provide at least two assets with amounts (e.g., AAPL 50000 TSLA 30000)"

                # -------- STEP 2: Convert to weights --------
                total = sum(holdings.values())
                weights = {k: v / total for k, v in holdings.items()}
                tickers = list(holdings.keys())

                # -------- STEP 3: Fetch data --------
                price_data = {}

                for t in tickers:
                    data = yf.download(t, period="6mo", auto_adjust=True, progress=False)

                    if data.empty:
                        return f"Invalid ticker: {t}"

                    close_series = data["Close"].squeeze()

                    if close_series is None or len(close_series) < 30:
                        return f"Not enough data for {t}"

                    price_data[t] = close_series

                # -------- STEP 4: Evaluate --------
                result = self.risk_service.evaluate_portfolio_full(
                    price_dict=price_data,
                    weights=weights
                )

                return result["explanation"]

            except Exception as e:
                return f"Portfolio error: {str(e)}"

        if "risk" in q_lower and target:

            price_series = self.fetcher.get_historical_prices(target)

            if price_series is None or len(price_series) < 30:
                return f"Not enough historical data for {target}"

            predicted_prices = price_series.tail(30).values

            ctx = self.risk_service.full_analysis(
                ticker=target,
                price_series=price_series,
                predicted_prices=predicted_prices
            )

            return ctx.explanation

        # ---------------- Stock price ----------------
        if intent == "get stock price" and target:
            reply = self.fetcher.get_stock_price(target)
            self.last_intent, self.last_target = intent, target
            self.kb.add_entry(f"Q: {question}\nA: {reply}")
            return reply

        # ---------------- Company news ----------------
        if intent == "get company news" and target:
            # use combined company + related general news
            reply = self.fetcher.get_company_and_related_news(
                target,
                company_names=target_company_keywords,
                limit=6,  # show up to 6 combined items
                summarize=True,
                model=self.model,
                tokenizer=self.tokenizer
            )
            self.last_intent, self.last_target = intent, target
            self.kb.add_entry(f"Q: {question}\nA: {reply}")
            return reply
    
        # ---------------- Global / category news ----------------
        if intent == "get global financial news":
            # Detect specific topic requests (energy, politics, tech, environment, geo)
            category_filter = None
            for keyword, cat in {
                "oil": "energy",
                "energy": "energy",
                "politic": "politics",
                "politics": "politics",
                "tech": "technology",
                "ai": "technology",
                "environment": "environment",
                "climate": "environment",
                "geo": "geopolitics",
                "india": "geopolitics",
                "china": "geopolitics",
                "us": "geopolitics",
            }.items():
                if keyword in q_lower:
                    category_filter = cat
                    break

            reply = self.fetcher.get_market_impact_news(
                limit_per_category=3,
                summarize=True,
                model=self.model,
                tokenizer=self.tokenizer,
                category_filter=category_filter
            )
            self.last_intent, self.last_target = intent, category_filter or "market-impact"
            self.kb.add_entry(f"Q: {question}\nA: {reply}")
            return reply
        

        # ---------------- Default LLM reasoning ----------------
        reply = self.generate_advice(question)

        # store only useful answers (avoid garbage)
        if len(reply) > 50 and "risk" not in reply.lower():
            self.kb.add_entry(f"Q: {question}\nA: {reply}")

        return reply


# ===========================
# 🚀 6. Run the Bot
# ===========================
if __name__ == "__main__":
    FINNHUB_API_KEY = "d2vhh09r01qm5lo8bhb0d2vhh09r01qm5lo8bhbg"  # replace with your real API key

    kb = KnowledgeBase()
    fetcher = FinancialDataFetcher(finhub_api_key=FINNHUB_API_KEY)
    bot = FinanceBot(kb, fetcher)

    print("\n Welcome to FinanceBot — your conversational AI financial advisor!")
    print("Ask about stock prices, financial news (company or global), or investment advice.")
    print("Type 'exit' anytime to quit.\n")

    while True:
        q = input("🧍‍♂️ You: ").strip()
        if q.lower() in ["exit", "quit", "bye"]:
            print("FinanceBot: Goodbye! Stay financially sharp ")
            break
        print(f"FinanceBot: {bot.answer(q)}\n")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Financial Advisor Model...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

C:\Users\shahd\AppData\Local\Temp\ipykernel_7564\1192519295.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(


FinanceBot: 
        Full Risk Analysis for TSLA:

        Statistical Risk:
        - Volatility: 0.0274
        - Expected Shortfall: -0.0564
        - Risk Score: 0.0390

        Market Regime:
        - Regime: normal_vol
        - Adjusted Risk: 1.00%

        

        Composite Risk:
        - Score: 0.0711
        - Level: High Risk

        Recommendation:
        High statistical risk. Limit exposure.
        



c:\Users\shahd\Desktop\hybrid_financial_chatbot\core\risk_sentinel.py:38: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "volatility": float(vol),
c:\Users\shahd\Desktop\hybrid_financial_chatbot\core\risk_sentinel.py:39: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "expected_shortfall": float(es),
c:\Users\shahd\Desktop\hybrid_financial_chatbot\core\risk_sentinel.py:40: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "risk_score": float(risk_score)
c:\Users\shahd\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\shahd\AppData\Local\Programs\Py

FinanceBot: I don't have enough data to answer that question.

FinanceBot: 
    Portfolio Risk Analysis:

    - Portfolio Volatility: 0.0116

    Risk Contribution:
    - AAPL: 81.65% (Risk Score: 0.02)
- MSFT: 18.35% (Risk Score: 0.03)

Interpretation:
- Portfolio is relatively STABLE

Recommendations:
- AAPL contributes most to risk (81.65%)
- Reduce AAPL exposure by ~51.0%
- Diversification can reduce portfolio risk




ValueError: You must include at least one label and at least one sequence.